
# Explainable ML-Based Classification of Modern Encrypted VPN Traffic Using Flow-Level Network Metadata

## 1. Dataset Audit

### Objective
The purpose of this notebook is to inspect the structure and quality of the available VPN traffic dataset before feature engineering and machine learning.

### Research Goals
- Classify Normal (Non-VPN) traffic versus VPN traffic.
- Investigate classification between OpenVPN and WireGuard.
- Evaluate generalization across historical and modern datasets.
- Explore explainable machine learning using SHAP.

### Current Audit Scope
- Dataset: 2022 JSON dataset
- Protocol folder: VPN / OpenVPN
- Application category: SSH
- File: `ssh.json`

### Initial Findings
- Flow records inspected: 5
- Packet entries inspected: 456
- Top-level fields: `ip_proto`, `port_dst`, `port_src`, `x_packets`
- Missing values in the checked packet fields: 0
- Negative `bytes` values: 234
- Unparseable `bytes` values: 0

### Important Limitations
These findings apply only to the small OpenVPN/SSH sample file, not the complete dataset. The meaning of negative byte values has not yet been verified. No values have been modified, and no machine learning models have been trained.


In [1]:

import json
from pathlib import Path
from collections import Counter

# Path to the small sample file
data_path = (
    Path.home()
    / "vpn-traffic-fyp/data/raw/extracted/2022_json/"
      "encrypted_vpn_dataset/VPN/OpenVPN/ssh.json"
)

print("File exists:", data_path.exists())
print("File size (KB):", round(data_path.stat().st_size / 1024, 2))

with data_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

print("Flow records:", len(records))

packet_counts = [len(r.get("x_packets", [])) for r in records]
print("Packet counts per flow:", packet_counts)
print("Total packet entries:", sum(packet_counts))

print("Top-level fields:", list(records[0].keys()))

missing = Counter()
negative_bytes = 0
unparseable_bytes = 0

for record in records:
    for packet in record.get("x_packets", []):
        for field in (
            "bytes", "timestamp_start", "timestamp_end",
            "ip_header_len", "tcp_header_len", "tcp_flags"
        ):
            if packet.get(field) is None:
                missing[field] += 1

        try:
            if int(packet["bytes"]) < 0:
                negative_bytes += 1
        except (ValueError, TypeError, KeyError):
            unparseable_bytes += 1

print("Missing values:", dict(missing))
print("Negative bytes values:", negative_bytes)
print("Unparseable bytes values:", unparseable_bytes)


File exists: True
File size (KB): 152.38
Flow records: 5
Packet counts per flow: [39, 104, 50, 101, 162]
Total packet entries: 456
Top-level fields: ['ip_proto', 'port_dst', 'port_src', 'x_packets']
Missing values: {}
Negative bytes values: 234
Unparseable bytes values: 0


In [2]:

import pandas as pd

flow_summary = []

for i, record in enumerate(records, start=1):
    values = []

    for packet in record.get("x_packets", []):
        try:
            values.append(int(packet["bytes"]))
        except (ValueError, TypeError, KeyError):
            pass

    flow_summary.append({
        "flow_id": i,
        "protocol": record.get("ip_proto"),
        "source_port": record.get("port_src"),
        "destination_port": record.get("port_dst"),
        "packet_entries": len(record.get("x_packets", [])),
        "negative_bytes": sum(v < 0 for v in values),
        "zero_bytes": sum(v == 0 for v in values),
        "positive_bytes": sum(v > 0 for v in values),
        "minimum_bytes": min(values) if values else None,
        "maximum_bytes": max(values) if values else None,
    })

flow_df = pd.DataFrame(flow_summary)
flow_df


,flow_id,protocol,source_port,destination_port,packet_entries,negative_bytes,zero_bytes,positive_bytes,minimum_bytes,maximum_bytes
0,1,tcp,46084,1194,39,20,0,19,-1223,1335
1,2,tcp,1194,46084,104,51,0,53,-1458,1591
2,3,tcp,46084,1194,50,26,0,24,-1223,1335
3,4,tcp,46084,1194,101,53,0,48,-1223,1335
4,5,tcp,46084,1194,162,84,0,78,-1591,1591



## 2. Verified Dataset Documentation

### Source
Naas, M., & Fesl, J. (2023). *A novel dataset for encrypted virtual private network traffic analysis*. Data in Brief, 47, 108945.

- Paper: https://doi.org/10.1016/j.dib.2023.108945
- Dataset: USBVPN2022
- Dataset repository: https://zenodo.org/record/7301756

### Verified Meaning of the `bytes` Field
According to Table 3 of the paper, `bytes` represents packet payload size with a direction sign:
- Positive values indicate forward-direction packets.
- Negative values indicate backward-direction packets.

### Research Decision
Negative values must not be treated as invalid data or removed automatically. During feature engineering, packet payload size and traffic direction will be handled separately where appropriate, while preserving the original values.

### Evidence from the Initial Sample
- Total packet entries: 456
- Forward-direction entries (positive): 222
- Backward-direction entries (negative): 234
- Zero entries: 0

These statistics apply only to the inspected OpenVPN/SSH sample.


## 3. Full Dataset Audit Results

### Objective
Perform a file-level structural and data-quality audit of the extracted 2022 JSON dataset before feature engineering or model training.

### Method
- Scan the extracted JSON files while excluding Jupyter checkpoint files.
- Parse records incrementally rather than loading entire large files into memory.
- Count flow records and packet records.
- Inspect byte-value signs, timestamp fields, packet-list availability, and flow-record structure.
- Save the per-file results to `results/2022_json_audit.csv`.

The audit script is `src/audit_json.py`. The source dataset files are read-only inputs; the script writes a separate report.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("results/2022_json_audit.csv"),
    Path("../results/2022_json_audit.csv"),
]
report_path = next((p for p in candidates if p.exists()), None)

if report_path is None:
    raise FileNotFoundError(
        "Audit CSV not found. Run the notebook from the project root "
        "or check that results/2022_json_audit.csv exists."
    )

audit = pd.read_csv(report_path)

count_columns = [
    "flows", "packet_records", "positive_bytes", "negative_bytes",
    "zero_bytes", "missing_or_invalid_bytes", "invalid_timestamp_fields",
    "flows_without_packet_list", "non_dict_flows"
]
for col in count_columns:
    if col in audit.columns:
        audit[col] = pd.to_numeric(audit[col], errors="coerce").fillna(0)

checks = [
    "missing_or_invalid_bytes",
    "invalid_timestamp_fields",
    "flows_without_packet_list",
    "non_dict_flows",
]
flags = audit[checks]
flagged_files = flags.gt(0).any(axis=1)

print(f"Report: {report_path}")
print(f"Files audited: {len(audit):,}")
print(f"Files with status OK: {(audit['status'] == 'OK').sum():,}")
print(f"Files with non-OK status: {(audit['status'] != 'OK').sum():,}")
print(f"Total flow records: {audit['flows'].sum():,.0f}")
print(f"Total packet records: {audit['packet_records'].sum():,.0f}")

for col in [
    "positive_bytes", "negative_bytes", "zero_bytes",
    "missing_or_invalid_bytes", "invalid_timestamp_fields",
    "flows_without_packet_list", "non_dict_flows"
]:
    print(f"{col}: {audit[col].sum():,.0f}")

print(f"Files with one or more quality flags: {flagged_files.sum():,}")

display(audit[[
    "relative_path", "flows", "packet_records",
    "missing_or_invalid_bytes", "invalid_timestamp_fields", "status"
]])

### Verified findings

The completed local audit covered **35 JSON files**. All 35 reported `OK` status, with no file-level parsing errors.

| Measure | Result |
|---|---:|
| Flow records | 58,977 |
| Packet records | 83,936,023 |
| Records with positive byte values | 36,211,120 |
| Records with negative byte values | 47,724,903 |
| Records with zero byte values | 0 |
| Missing or invalid byte values | 0 |
| Invalid timestamp fields | 0 |
| Flows without a packet list | 0 |
| Non-dictionary flow records | 0 |

The four audited quality counters were zero across all files. Positive and negative counts refer to records grouped by the sign of the byte value; they are not byte-volume totals.

### Interpretation

These results support that the scanned files were structurally parseable and passed the specific checks implemented by the audit script. In this dataset, negative byte values must not automatically be treated as invalid: the dataset documentation describes the sign as an indicator of packet direction.

### Limitations and next steps

- A successful structural audit does not establish that the dataset is ready for machine learning.
- Labels, class balance, feature distributions, duplicate flows, potential leakage, and train/test separation still require investigation.
- The audit does not establish model accuracy or prove that models generalize to newer traffic.
- The results apply only to the audited 2022 JSON files, not to the 2016, 2026 Parquet, or future self-captured datasets.

**Next step:** verify the file-level labels and class distribution, then assess feature consistency and potential leakage before preparing any training data.